# Random Forest hyperparameter tuning

This notebook tunes the strongest classical baseline obtained so far:
Random Forest with the full 28-feature representation.

The hyperparameter search is performed using cross-validation on the
training set. The validation set is used only after the search to
evaluate the selected configuration.

The test set remains untouched.

In [ ]:
from pathlib import Path
import pickle
import sys
import time

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

sys.path.append(str(Path("../src").resolve()))

from rfml.features import build_feature_table

## 1. Data loading

In [ ]:

DATA_PATH = Path(
    "../data/raw/RML2016.10a/RML2016.10a_dict.pkl"
)

PROCESSED_DIR = Path("../data/processed")

with open(DATA_PATH, "rb") as f:
    radioml = pickle.load(f, encoding="latin1")

train_metadata = pd.read_csv(
    PROCESSED_DIR / "train_metadata.csv"
)

validation_metadata = pd.read_csv(
    PROCESSED_DIR / "validation_metadata.csv"
)

print("RadioML combinations:", len(radioml))
print("Train metadata:", train_metadata.shape)
print("Validation metadata:", validation_metadata.shape)

## 2. Tuning subset

For efficient experimentation, hyperparameter tuning is performed on
the same balanced subset used during model comparison:

- 50 samples per `(modulation, SNR)` stratum for training.
- 20 samples per stratum for validation.

In [ ]:
train_tuning = (
    train_metadata
    .groupby(["modulation", "snr"], group_keys=False)
    .sample(n=50, random_state=42)
    .reset_index(drop=True)
)

validation_tuning = (
    validation_metadata
    .groupby(["modulation", "snr"], group_keys=False)
    .sample(n=20, random_state=42)
    .reset_index(drop=True)
)

print("Train tuning:", train_tuning.shape)
print("Validation tuning:", validation_tuning.shape)

In [ ]:
train_full = build_feature_table(
    radioml,
    train_tuning,
    feature_set="full"
)

validation_full = build_feature_table(
    radioml,
    validation_tuning,
    feature_set="full"
)

metadata_columns = [
    "sample_id",
    "modulation",
    "snr"
]

feature_columns = [
    column
    for column in train_full.columns
    if column not in metadata_columns
]

X_train = train_full[feature_columns]
y_train = train_full["modulation"]

X_validation = validation_full[feature_columns]
y_validation = validation_full["modulation"]

print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)
print("Number of features:", len(feature_columns))

In [ ]:
baseline_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

baseline_rf.fit(
    X_train,
    y_train
)

baseline_predictions = baseline_rf.predict(
    X_validation
)

baseline_accuracy = accuracy_score(
    y_validation,
    baseline_predictions
)

baseline_f1 = f1_score(
    y_validation,
    baseline_predictions,
    average="macro"
)

print("Baseline Random Forest")
print(f"Accuracy: {baseline_accuracy:.4f}")
print(f"Macro-F1: {baseline_f1:.4f}")

In [ ]:
train_strata = (
    train_full["modulation"].astype(str)
    + "_"
    + train_full["snr"].astype(str)
)

In [ ]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

cv_splits = list(
    cv.split(
        X_train,
        train_strata
    )
)

print("Number of CV folds:", len(cv_splits))

In [ ]:
param_distributions = {
    "n_estimators": [150, 250, 400, 600],

    "max_depth": [
        10,
        15,
        20,
        30,
        None
    ],

    "min_samples_split": [
        2,
        5,
        10
    ],

    "min_samples_leaf": [
        1,
        2,
        4,
        8
    ],

    "max_features": [
        "sqrt",
        "log2",
        0.5,
        None
    ]
}

## 3. Randomized hyperparameter search

RandomizedSearchCV explores a limited number of combinations from the
hyperparameter space.

Macro-F1 is used as the optimization metric because all modulation
classes should contribute equally to model selection.

In [ ]:
rf_model = RandomForestClassifier(
    random_state=42,

    # RandomizedSearchCV handles the parallelization.
    n_jobs=1
)

rf_search = RandomizedSearchCV(
    estimator=rf_model,

    param_distributions=param_distributions,

    n_iter=20,

    scoring="f1_macro",

    cv=cv_splits,

    random_state=42,

    n_jobs=-1,

    verbose=1,

    refit=True
)

In [ ]:
start = time.perf_counter()

rf_search.fit(
    X_train,
    y_train
)

search_time = time.perf_counter() - start

print(
    f"Tuning time: {search_time:.2f} seconds"
)

In [ ]:
print("Best CV macro-F1:")
print(rf_search.best_score_)

print("\nBest parameters:")
print(rf_search.best_params_)

In [ ]:
best_rf = rf_search.best_estimator_

tuned_predictions = best_rf.predict(
    X_validation
)

tuned_accuracy = accuracy_score(
    y_validation,
    tuned_predictions
)

tuned_f1 = f1_score(
    y_validation,
    tuned_predictions,
    average="macro"
)

print("Tuned Random Forest")
print(f"Accuracy: {tuned_accuracy:.4f}")
print(f"Macro-F1: {tuned_f1:.4f}")

In [ ]:
tuning_results = pd.DataFrame([
    {
        "model": "RF baseline",
        "accuracy": baseline_accuracy,
        "macro_f1": baseline_f1,
    },
    {
        "model": "RF tuned",
        "accuracy": tuned_accuracy,
        "macro_f1": tuned_f1,
    },
])

tuning_results